# Simple Offline Flower Classification (VGG16)

This is a simpler offline version of your original notebook. The VGG16 model and Dense layers are kept. Only the online dataset loading is replaced with loading images from your computer.

**Dataset folder format:**
```text
flower_dataset/
    daisy/
    dandelion/
    roses/
    sunflowers/
    tulips/
```
Use the actual class folders supplied by your teacher.

In [ ]:
# 1. Imports
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras import layers, models
from tensorflow.keras.utils import load_img, img_to_array

In [ ]:
# 2. Load dataset from your computer (OFFLINE)
DATASET_PATH = r"C:\Users\YourName\Downloads\flower_dataset"

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(150, 150),
    batch_size=32,
    label_mode="categorical"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(150, 150),
    batch_size=32,
    label_mode="categorical"
)

class_names = train_ds.class_names
print("Classes:", class_names)
print("Number of classes:", len(class_names))

In [ ]:
# 3. VGG16 preprocessing
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y))

## Build the same VGG16 classifier

`weights=None` stops TensorFlow from downloading ImageNet weights. The model will learn from the local dataset.

In [ ]:
# 4. Load VGG16 (OFFLINE)
base_model = VGG16(
    weights=None,
    include_top=False,
    input_shape=(150, 150, 3)
)
base_model.trainable = True

In [ ]:
# 5. Same Dense layers as your original notebook
flatten_layer = layers.Flatten()
dense_layer_1 = layers.Dense(50, activation='relu')
dense_layer_2 = layers.Dense(20, activation='relu')
prediction_layer = layers.Dense(len(class_names), activation='softmax')

model = models.Sequential([
    base_model,
    flatten_layer,
    dense_layer_1,
    dense_layer_2,
    prediction_layer
])

In [ ]:
# 6. Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# 7. Train the model
history = model.fit(
    train_ds,
    epochs=1,
    validation_data=test_ds
)

In [ ]:
# 8. Evaluate the model
loss, accuracy = model.evaluate(test_ds)
print("Loss:", loss)
print("Accuracy:", accuracy)

In [ ]:
# 9. Predict test images and compare actual vs predicted
for images, labels in test_ds.take(1):
    predictions = model.predict(images, verbose=0)

    for i in range(min(5, len(images))):
        actual = class_names[np.argmax(labels[i].numpy())]
        predicted = class_names[np.argmax(predictions[i])]
        print("Actual:", actual, "| Predicted:", predicted)

        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title("Actual: " + actual + " | Predicted: " + predicted)
        plt.axis("off")
        plt.show()

In [ ]:
# 10. Predict any one image from your computer
IMAGE_PATH = r"C:\Users\YourName\Downloads\test_flower.jpg"

img = load_img(IMAGE_PATH, target_size=(150, 150))
plt.imshow(img)
plt.axis("off")
plt.show()

img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

prediction = model.predict(img_array, verbose=0)
predicted_class = class_names[np.argmax(prediction[0])]
confidence = np.max(prediction[0]) * 100

print("Predicted flower:", predicted_class)
print("Confidence:", round(float(confidence), 2), "%")

In [ ]:
# 11. Save model (optional)
model.save("flower_vgg16_offline.keras")
print("Model saved.")

## Remember
- Change only `DATASET_PATH` to the dataset folder location.
- Change `IMAGE_PATH` to the image you want to predict.
- This notebook does not download the dataset or VGG16 weights.
- Because `weights=None`, VGG16 trains from scratch. It may need more epochs and time for good accuracy.
- TensorFlow and required Python packages must already be installed before you disconnect from the internet.